# Comparing stocks

One page across every stock run with `python run.py`, built from the summaries each report saved in `reports/summaries/`. It has the same two parts as the reports:

- **Part 1, direction:** up or down tomorrow, judged against "always up."
- **Part 2, move size:** big or small move tomorrow, judged against "same as today."

**The lucky-winners problem.** Test enough stocks and models, and some will "win" by chance alone. At the usual 5% level, about 1 comparison in 20 passes even when nothing real is going on. So each part shows two bars:

- **5%:** the usual threshold for a single test.
- **Corrected:** 5% divided by the number of comparisons in that part (the *Bonferroni correction*). A result has to clear this to count as evidence once many things were tested. The two parts answer different questions, so each is corrected on its own.

In [ ]:
# Settings (run.py replaces these)
TICKERS = ["SPY"]

In [ ]:
import sys
from pathlib import Path

# Find the project folder, whether this runs from the project, notebooks/ or reports/
ROOT = Path.cwd()
if ROOT.name in ("notebooks", "reports"):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from IPython.display import Markdown, display

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (11, 4)
# Same colors in every report: blue, orange, aqua, yellow
plt.rcParams["axes.prop_cycle"] = cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])

import numpy as np
from src.report import comparison_table, load_summaries

summaries = load_summaries(TICKERS)
dir_table, dir_info = comparison_table(summaries, task="direction")
vol_table, vol_info = comparison_table(summaries, task="volatility")
print(f"Stocks: {', '.join(TICKERS)}")

MODEL_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def plot_edges(table, days, baseline, title):
    """Each model's accuracy minus the baseline's, with its 95% range."""
    plot = table.reset_index()
    models = list(dict.fromkeys(plot["model"]))
    stocks = list(dict.fromkeys(plot["stock"]))
    colors = dict(zip(models, MODEL_COLORS))
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(stocks) * len(models) + 1.5))
    y, ticks, labels = 0, [], []
    for stock in stocks:
        for model in models:
            row = plot[(plot["stock"] == stock) & (plot["model"] == model)].iloc[0]
            half = 1.96 * np.sqrt(row["accuracy"] * (1 - row["accuracy"]) / days[stock])
            ax.errorbar(row["edge"] * 100, y, xerr=half * 100, fmt="o", color=colors[model],
                        markersize=8, capsize=4, linewidth=2, label=model if stock == stocks[0] else None)
            ticks.append(y)
            labels.append(f"{stock}  {model}")
            y += 1
        y += 0.5
    ax.axvline(0, color="grey", linewidth=1)
    ax.set_yticks(ticks, labels)
    ax.invert_yaxis()
    ax.set_xlabel(f'accuracy minus "{baseline}" (percentage points)')
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=len(models), frameon=False)
    plt.tight_layout()
    plt.show()


def luck_summary(info, accuracy_test, luck_test):
    """How many passes there are, next to how many luck alone would produce."""
    n = info["comparisons"]
    models_per_stock = n // max(len(TICKERS), 1)
    display(Markdown(f"""
- **Comparisons:** {n} ({len(TICKERS)} stocks × {models_per_stock} models)
- **Expected passes by pure luck at 5%:** about {info["expected chance passes"]:.1f} per test
- **Corrected bar:** p-value or luck share below {info["corrected threshold"]:.4f}

| Test | Passes at 5% | Passes after correction |
|---|---|---|
| {accuracy_test} | {info["accuracy passes (5%)"]} of {n} | {info["accuracy passes (corrected)"]} of {n} |
| {luck_test} | {info["luck passes (5%)"]} of {n} | {info["luck passes (corrected)"]} of {n} |
"""))

# Part 1: direction

## 1. Test periods

Each stock is tested from its own first test year, so periods can differ for stocks with shorter histories.

In [ ]:
pd.DataFrame([
    {"stock": s["ticker"], "test start": s["test start"], "test end": s["test end"],
     "test days": s["test days"], "always up accuracy": s["always up accuracy"],
     "buy and hold annual return": s["buy and hold"]["annual return"]}
    for s in summaries
]).set_index("stock")

## 2. Accuracy against "always up"

**Edge** is the model's accuracy minus the "always up" accuracy for the same stock and days. Zero means no better than always predicting up. The bars show the 95% range: if a bar crosses zero, the edge could be luck.

In [ ]:
plot_edges(dir_table, {s["ticker"]: s["test days"] for s in summaries},
           "always up", 'Direction: model edge over "always up", with 95% range')

## 3. Full table

- **p-value vs always up:** the chance of an accuracy this high if the model were no better than "always up."
- **share of random >=:** from the luck check: how often random cash days did at least as well in the backtest, on annual return. Empty when a model never left the market.
- The four True/False columns: does the result pass at 5%, and does it pass the corrected bar?

In [ ]:
dir_table

## 4. How many wins would luck produce?

In [ ]:
luck_summary(dir_info, 'Accuracy beats "always up"', "Backtest beats luck (annual return)")
display(Markdown("A handful of passes at 5%, close to what luck predicts, is **not** evidence of skill. "
                 "Only results that also pass the corrected bar, and that hold up on future data "
                 "(phase 6, the live forward test), deserve attention."))

# Part 2: move size

## 5. Baselines and buy-and-hold for each stock

- **share of big days:** how often tomorrow's move beat the stock's typical day over the past year.
- **majority:** accuracy of always predicting whichever is more common.
- **same as today:** accuracy of predicting tomorrow will be big if today was big. This is the bar for Part 2.
- **buy and hold:** the Sharpe ratio and worst drop that the half-on-big strategy tries to improve on.

In [ ]:
pd.DataFrame([
    {"stock": s["ticker"], "test start": s["volatility"]["test start"], "test days": s["volatility"]["test days"],
     "share of big days": s["volatility"]["share of big days"],
     "majority": s["volatility"]["majority accuracy"],
     "same as today": s["volatility"]["same as today accuracy"],
     "buy and hold Sharpe": s["volatility"]["buy and hold"]["Sharpe ratio"],
     "buy and hold max drawdown": s["volatility"]["buy and hold"]["max drawdown"]}
    for s in summaries
]).set_index("stock")

## 6. Accuracy against "same as today"

**Edge** is the model's accuracy minus the "same as today" accuracy. Bars entirely right of zero mean the model reliably beats the one-line rule.

In [ ]:
plot_edges(vol_table, {s["ticker"]: s["volatility"]["test days"] for s in summaries},
           "same as today", 'Move size: model edge over "same as today", with 95% range')

## 7. Full table

- **p-value vs same as today:** the chance of an accuracy this high if the model were no better than "same as today."
- **Sharpe / max drawdown (half on big):** the strategy that holds a full position on predicted calm days and half on predicted big days, after trading costs, next to buy-and-hold.
- **share of random >=:** from the luck check: how often random half-position days reached at least the same Sharpe ratio.

In [ ]:
vol_table

## 8. How many wins would luck produce?

In [ ]:
luck_summary(vol_info, 'Accuracy beats "same as today"', "Half-on-big beats luck (Sharpe ratio)")

## 9. Conclusions

*Write this in your own words.* Some prompts:

- **Direction:** did any stock show a real edge after the correction? Were results consistent across stocks?
- **Move size:** did the models beat "same as today" on every stock, and after the correction?
- Did predicting move size improve the Sharpe ratio or the worst drop, after trading costs?
- How much did buy-and-hold differ between stocks, and how much of that is hindsight from picking well-known winners?